In [3]:
# Import the required libraries.
import tkinter as tk
from tkinter import messagebox
from tkinter.scrolledtext import ScrolledText

import pandas as pd
import numpy as np
import joblib
import os

from tensorflow.keras.models import load_model


# Load the trained Tuned DNN model and TF-IDF vectorizer.


model = load_model("best_dnn_model.keras")
tfidf = joblib.load("tfidf.pkl")


# Create the feedback file if it does not exist.

feedback_file = "human_evaluation_feedback.csv"

if not os.path.exists(feedback_file):

    feedback_df = pd.DataFrame(columns=[
        "Email",
        "Predicted Class",
        "Confidence (%)",
        "Human Feedback"
    ])

    feedback_df.to_csv(feedback_file, index=False)

# Predict Email Function.


def predict_email():

    # I retrieved the email entered by the user.
    email = txt_email.get("1.0", tk.END).strip()

    # I checked whether the email is empty.
    if email == "":

        messagebox.showwarning(
            "Input Required",
            "Please enter an email."
        )

        return

    # I converted the email into TF-IDF features.
    email_vector = tfidf.transform([email])

    # I converted the sparse matrix into a dense array.
    email_vector = email_vector.toarray()

    # I predicted the probability using the Tuned DNN.
    probability = model.predict(email_vector, verbose=0)[0][0]

    # I converted the probability into the predicted class.
    if probability >= 0.5:

        prediction = 1
        predicted_class = "Spam"
        confidence = probability * 100

    else:

        prediction = 0
        predicted_class = "Non-Spam"
        confidence = (1 - probability) * 100

    # I displayed the prediction.
    lbl_prediction.config(
        text=f"Prediction : {predicted_class}"
    )

    # I displayed the confidence score.
    lbl_confidence.config(
        text=f"Confidence : {confidence:.2f}%"
    )


# Save Human Feedback.

def save_feedback(feedback):

    # I retrieved the email.
    email = txt_email.get("1.0", tk.END).strip()

    # I checked whether prediction has been performed.
    if lbl_prediction.cget("text") == "Prediction : ":

        messagebox.showwarning(
            "Prediction Required",
            "Please predict an email first."
        )

        return

    prediction = lbl_prediction.cget("text").replace("Prediction : ", "")

    confidence = lbl_confidence.cget("text").replace("Confidence : ", "")

    # I created a new feedback row.
    row = pd.DataFrame({

        "Email":[email],

        "Predicted Class":[prediction],

        "Confidence (%)":[confidence],

        "Human Feedback":[feedback]

    })

    # I appended the feedback into the CSV file.
    row.to_csv(

        feedback_file,

        mode="a",

        header=False,

        index=False

    )

    messagebox.showinfo(

        "Feedback Saved",

        "Thank you for your feedback."

    )


# Clear Function.


def clear_all():

    txt_email.delete("1.0", tk.END)

    lbl_prediction.config(text="Prediction : ")

    lbl_confidence.config(text="Confidence : ")


# Create Main Window.


window = tk.Tk()

window.title("Email Spam Classification Prototype")

window.geometry("950x720")

window.configure(bg="white")


# Title.


title = tk.Label(

    window,

    text="Email Spam Classification Prototype",

    font=("Arial",22,"bold"),

    bg="white"

)

title.pack(pady=15)


# Email Input.


label = tk.Label(

    window,

    text="Enter an Email",

    font=("Arial",14),

    bg="white"

)

label.pack()

txt_email = ScrolledText(

    window,

    width=100,

    height=15,

    font=("Arial",11)

)

txt_email.pack(pady=10)


# Buttons.

button_frame = tk.Frame(window,bg="white")

button_frame.pack(pady=10)

predict_button = tk.Button(

    button_frame,

    text="Predict",

    width=18,

    font=("Arial",12),

    command=predict_email

)

predict_button.grid(row=0,column=0,padx=10)

clear_button = tk.Button(

    button_frame,

    text="Clear",

    width=18,

    font=("Arial",12),

    command=clear_all

)

clear_button.grid(row=0,column=1,padx=10)


# Prediction Result.


lbl_prediction = tk.Label(

    window,

    text="Prediction : ",

    font=("Arial",16,"bold"),

    bg="white"

)

lbl_prediction.pack(pady=10)

lbl_confidence = tk.Label(

    window,

    text="Confidence : ",

    font=("Arial",16,"bold"),

    bg="white"

)

lbl_confidence.pack()


# Human Evaluation Section.


feedback_title = tk.Label(

    window,

    text="Human Evaluation",

    font=("Arial",18,"bold"),

    bg="white"

)

feedback_title.pack(pady=20)

feedback_label = tk.Label(

    window,

    text="Please verify whether the prediction is correct.",

    font=("Arial",12),

    bg="white"

)

feedback_label.pack()

feedback_frame = tk.Frame(window,bg="white")

feedback_frame.pack(pady=15)

correct_button = tk.Button(

    feedback_frame,

    text="Prediction is Correct",

    width=25,

    font=("Arial",12),

    command=lambda: save_feedback("Correct")

)

correct_button.grid(row=0,column=0,padx=15)

incorrect_button = tk.Button(

    feedback_frame,

    text="Prediction is Incorrect",

    width=25,

    font=("Arial",12),

    command=lambda: save_feedback("Incorrect")

)

incorrect_button.grid(row=0,column=1,padx=15)

# Exit Button.


exit_button = tk.Button(

    window,

    text="Exit",

    width=20,

    font=("Arial",12),

    command=window.destroy

)

exit_button.pack(pady=25)


# Run the GUI.

window.mainloop()